# 01 · Demanda demográfica — IBGE 2022

**Objetivo.** Transformar os setores censitários de Lavras em uma estimativa espacial de demanda escolar, **sem fazer a soma igualar as matrículas**. Este notebook adapta a lógica útil dos notebooks de AED existentes: auditoria dos registros, geometria, grupos etários, supressões e análises por setor.

A variável \(A_{ie}\) é a população da faixa etária de referência para a etapa \(e\), não o número observado de alunos dessa etapa. Estimamos \(w_{ie}(\rho)=\operatorname{arredondamento}(\rho_e A_{ie})\), onde \(\rho_e\) é uma **hipótese de cenário** de participação na rede pública. Os grupos etários são **5–9, 10–14 e 15–19 anos**, aproximados às etapas iniciais, finais e Médio, respectivamente; isso é um proxy e não uma correspondência aluno/série.

Os valores demográficos suprimidos (`X`) são preservados em flags e imputados separadamente. Para testar robustez, comparar os cenários e relatar a parcela imputada.

## 1. Carregamento e inspeção das fontes

A malha pode conter um setor em múltiplas partes geométricas. Unimos essas partes antes de produzir pontos representativos. Todas as coordenadas são armazenadas em projeção UTM local, permitindo distâncias em quilômetros.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

raiz = Path.cwd().resolve()
if not (raiz / "pyproject.toml").exists():
    raiz = next((p for p in raiz.parents if (p / "pyproject.toml").exists()), raiz)
assert (raiz / "pyproject.toml").exists(), "Execute dentro do repositório PM."
saida = raiz / "outputs" / "localizacao_escolar" / "lavras"
saida.mkdir(parents=True, exist_ok=True)
CODIGO_MUNICIPIO = "3138203"
ETAPAS = {"fundamental_iniciais": ("V01032", "QT_MAT_FUND_AI", "QT_TUR_FUND_AI"),
          "fundamental_finais": ("V01033", "QT_MAT_FUND_AF", "QT_TUR_FUND_AF"),
          "medio": ("V01034", "QT_MAT_MED_PROP", "QT_TUR_MED_PROP")}
import geopandas as gpd
import matplotlib.pyplot as plt
ARQUIVO_MALHA = raiz / "data" / "BR_setores_CD2022.gpkg"
ARQUIVO_IDADES = raiz / "data" / "Agregados_por_setores_demografia_BR.csv"
for caminho in (ARQUIVO_MALHA, ARQUIVO_IDADES):
    assert caminho.exists(), f"Base ausente: {caminho}. Consulte o README."

In [ ]:
malha = gpd.read_file(ARQUIVO_MALHA, layer="BR_setores_CD2022",
                     where=f"CD_MUN = '{CODIGO_MUNICIPIO}'",
                     columns=["CD_SETOR", "CD_MUN", "SITUACAO", "v0001"])
assert len(malha) > 0
assert malha.CD_SETOR.notna().all() and malha.geometry.notna().all()
assert malha.geometry.is_valid.all(), "Inspecione geometrias inválidas antes da união"
assert not malha.groupby("CD_SETOR")["v0001"].nunique(dropna=False).gt(1).any()
setores_geo = malha.dissolve(by="CD_SETOR", as_index=False)
assert setores_geo.CD_SETOR.is_unique
setores_geo = setores_geo.to_crs(setores_geo.estimate_utm_crs())
crs_utm = setores_geo.crs
representantes = setores_geo.geometry.representative_point()
setores_geo["x_m"] = representantes.x
setores_geo["y_m"] = representantes.y
print("Registros malha:", len(malha), "| setores únicos:", len(setores_geo), "| CRS:", crs_utm)

## 2. Faixas etárias e supressões do Censo

In [ ]:
COLUNAS = ["CD_setor", "V01006", "V01032", "V01033", "V01034"]
fatias = pd.read_csv(ARQUIVO_IDADES, sep=";", dtype="string", usecols=COLUNAS,
                     chunksize=50_000)
idades = pd.concat([p.loc[p.CD_setor.str[:7].eq(CODIGO_MUNICIPIO)] for p in fatias],
                  ignore_index=True).rename(columns={"CD_setor": "CD_SETOR"})
assert idades.CD_SETOR.is_unique and len(idades) > 0
for campo in COLUNAS[1:]:
    original = idades[campo].str.strip().str.upper()
    idades[campo + "_suprimido"] = original.eq("X").fillna(False)
    idades[campo] = pd.to_numeric(original.mask(original.eq("X")), errors="raise")
base = setores_geo.drop(columns="geometry").merge(idades, on="CD_SETOR", how="left",
                                                   validate="one_to_one", indicator=True)
assert base._merge.eq("both").all(), "Há setores da malha sem demografia"
base = base.drop(columns="_merge")
assert base.v0001.notna().all() and base.v0001.ge(0).all()
presentes = base.V01006.notna()
assert (base.loc[presentes, "V01006"] == base.loc[presentes, "v0001"]).all()
display(base[["CD_SETOR", "v0001", "V01032", "V01033", "V01034"]].head())
display(pd.DataFrame({"faixa": ["5-9", "10-14", "15-19"],
                      "suprimidos": [int(base[e[0] + "_suprimido"].sum()) for e in ETAPAS.values()]}))

## 3. Estimação transparente e cenários

Para cada faixa, valores faltantes são imputados como população total do setor multiplicada pela proporção etária observada nos setores não suprimidos. Em vez de calibrar por matrículas, usamos taxas de participação pública \(\rho\) **meramente ilustrativas**. As três taxas não foram estimadas por município e não significam cobertura oficial. O caso central é \(\rho=0{,}80\) e os extremos funcionam como sensibilidade.

A função de maiores restos preserva o total inteiro arredondado de \(\rho\sum_i A_{ie}\). O arredondamento não elimina a incerteza da demanda real.

In [ ]:
def maiores_restos(valores, total):
    valores = np.asarray(valores, dtype=float)
    assert (valores >= 0).all() and np.isfinite(valores).all()
    if valores.sum() == 0:
        assert total == 0
        return np.zeros(len(valores), dtype=int)
    ideal = valores * (total / valores.sum())
    inteiro = np.floor(ideal).astype(int)
    faltam = int(total - inteiro.sum())
    ordem = np.argsort(-(ideal - inteiro), kind="stable")
    inteiro[ordem[:faltam]] += 1
    assert inteiro.sum() == total
    return inteiro

cenario_participacao = {"baixa": 0.65, "central": 0.80, "alta": 0.95}
resumo = []
for etapa, (campo, _, _) in ETAPAS.items():
    conhecidos = base[campo].notna()
    denom = float(base.loc[conhecidos, "v0001"].sum())
    assert denom > 0
    taxa_faixa = float(base.loc[conhecidos, campo].sum() / denom)
    estimativa = base[campo].astype(float).fillna(base.v0001 * taxa_faixa)
    assert np.isfinite(estimativa).all() and (estimativa >= 0).all()
    base["idade_estimada_" + etapa] = estimativa
    for nome, rho in cenario_participacao.items():
        esperado = rho * estimativa
        total = int(np.floor(esperado.sum() + 0.5))
        base[f"demanda_{etapa}_{nome}"] = maiores_restos(esperado, total)
        resumo.append({"etapa": etapa, "cenario": nome, "rho_hipotetico": rho,
                       "demanda_total": total,
                       "setores_demanda_positiva": int((base[f"demanda_{etapa}_{nome}"] > 0).sum()),
                       "setores_com_idade_suprimida": int(base[campo + "_suprimido"].sum()),
                       "participacao_imputada_faixa": float(estimativa[~conhecidos].sum() / estimativa.sum())})
resumo = pd.DataFrame(resumo)
display(resumo)

## 4. AED e exportação

Verifique como a demanda varia por etapa, qual proporção dos setores exige imputação e a relação entre urbano/rural. Não use a demanda central como verdade observada.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
base["v0001"].hist(ax=axes[0], bins=25)
axes[0].set(title="População por setor", xlabel="Pessoas", ylabel="Setores")
for etapa in ETAPAS:
    base[f"demanda_{etapa}_central"].hist(ax=axes[1], bins=25, alpha=.45, label=etapa)
axes[1].set(title="Demanda hipotética por setor (rho=0,80)", xlabel="Estudantes estimados")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
base.to_csv(saida / "setores_demanda.csv", index=False)
resumo.to_csv(saida / "auditoria_demanda.csv", index=False)
(saida / "metadados_ibge.json").write_text(json.dumps({
    "ibge": 2022, "municipio": CODIGO_MUNICIPIO, "crs_projetado": crs_utm.to_string(),
    "coortes_proxy": {e: campos[0] for e, campos in ETAPAS.items()},
    "participacao_publica_hipotetica": cenario_participacao,
    "imputacao": "taxa etaria observada entre setores nao suprimidos aplicada a populacao do setor",
    "coordenadas_setores": "pontos representativos internos aos poligonos, nao residencias"
}, ensure_ascii=False, indent=2))
print("Exportado:", saida / "setores_demanda.csv")